# Proposed pipeline — Finding-Conditioned CT Temporal Progression (Colab)

Implements the design in `docs/proposed_pipeline.png`:

**d_f = g(v_prior, v_current, finding)**

**What is new vs `train_ctclip_temporal_colab.ipynb` (current pipeline):**
1. **Finding conditioning** inside the Difference Transformer (`e_diff <- e_diff + e_f`, or finding as a 4th token) so one pair produces many finding-specific `d_f`.
2. **Masked same-finding cross-modal SupCon** — image temporal rep `d_f` aligns to **frozen temporal-sentence embeddings** (dynamic/evidence text; synthetic stable templates only when needed). Positives = same finding + same direction texts; negatives = same finding + other direction; **other findings ignored**. Not instance CLIP InfoNCE; not `d`↔`d` only.
3. **Separate contrastive temperature** `tau_con` (does not share CE `logit_scale`).
4. **Contrastive-aware batching** (~K findings × 3 classes) so each ba usable positives/negatives.
5. **Ablation knobs:** CE | +mag | +text-SupCon · one-way vs symmetric · templates vs real prototypes · +/- finding conditioning.

**Unchanged protocol:**
- Frozen CT-CLIP image/text towers; only the Difference Transformer trains.
- Hub `train_*` → train + patient-held-out tune (early-stop on macro-F1); Hub `valid_*` → **one** final test.
- Inference: `argmax cos(d_f, PROTO[finding])` — **no report text** at test time.
- Prefer `report_explicit` / tier=`explicit` silver labels.

Diagram: `docs/proposed_pipeline.png`.

## 1. Setup: clone CT-CLIP + our repo, install deps

In [ ]:
%cd /content
![ -d CT-CLIP ] || git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git
%cd /content/CT-CLIP
!pip install -q -e transformer_maskgit
!pip install -q -e CT_CLIP
!pip install -q nibabel scipy huggingface_hub transformers scikit-learn tqdm
%cd /content
import sys
for p in ['/content/CT-CLIP/CT_CLIP', '/content/CT-CLIP/transformer_maskgit']:
    if p not in sys.path: sys.path.insert(0, p)
![ -d 3dCT ] || git clone https://github.com/nprakash1/3dCT.git 3dCT
!cd 3dCT && git pull -q
sys.path.append('/content/3dCT/scripts')
import ct_clip, transformer_maskgit; print('CT-CLIP import OK')

## 2. Mount Drive + config paths + ablation knobs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, csv, math, random, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict
from sklearn.metrics import f1_score, confusion_matrix
csv.field_size_limit(10**9)

DRIVE      = '/content/drive/MyDrive/3dCT'
IMG_DIR    = f'{DRIVE}/ctclip_cache/img'
WEIGHTS    = f'{DRIVE}/ctclip_weights'
PROTO_PT   = f'{DRIVE}/ctclip_cache/proto_bank.pt'
TTEXT_PT   = f'{DRIVE}/ctclip_cache/temporal_text_emb.pt'
LAB        = '/content/3dCT/medgemma_labels_v3.jsonl'
LAB_DS     = '/content/3dCT/medgemma_labels (2).jsonl'

TUNE_FRAC  = 0.15
SPLIT_SEED = 2026
REQUIRE_COMPLETE_HUB_VALID_FEATURES = True

CLASSES = ['worsened', 'stable', 'improved']
C2I = {c: i for i, c in enumerate(CLASSES)}
I2C = {i: c for c, i in C2I.items()}

FINDING_CONDITIONING = True
FINDING_AS_4TH_TOKEN = False
USE_LEARNED_FINDING_EMB = True

USE_CE         = True
USE_MAGNITUDE  = True
USE_SUPCON     = True

CONTRASTIVE_SYMMETRIC = False
LEARNABLE_TAU_CON     = True
TAU_CON_INIT          = 0.07
STABLE_TEXT_SEED      = 2026

PROTO_SOURCE   = 'real'
ANTISYM        = False

D_MODEL, EPOCHS, LR, PATIENCE = 256, 120, 1e-3, 20
WEIGHT_DECAY = 1e-2
LAMBDA_CE, LAMBDA_MAG, LAMBDA_CON = 1.0, 0.5, 0.5

K_FINDINGS_PER_BATCH = 8
N_PER_CLASS          = 4
MAX_BATCH_SIZE       = 256

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| img cache exists:', os.path.isdir(IMG_DIR))
print('knobs:', dict(
    FINDING_CONDITIONING=FINDING_CONDITIONING,
    FINDING_AS_4TH_TOKEN=FINDING_AS_4TH_TOKEN,
    USE_CE=USE_CE, USE_MAGNITUDE=USE_MAGNITUDE, USE_SUPCON=USE_SUPCON,
    CONTRASTIVE_SYMMETRIC=CONTRASTIVE_SYMMETRIC,
    LEARNABLE_TAU_CON=LEARNABLE_TAU_CON,
    PROTO_SOURCE=PROTO_SOURCE, ANTISYM=ANTISYM,
    LAMBDA_CE=LAMBDA_CE, LAMBDA_MAG=LAMBDA_MAG, LAMBDA_CON=LAMBDA_CON,
    TAU_CON_INIT=TAU_CON_INIT))

## 3. Load cached CT-CLIP IMAGE embeddings (512-d per volume)

In [ ]:
import glob
POOLED = {}
for fp in glob.glob(f'{IMG_DIR}/*.pt'):
    key = os.path.basename(fp)[:-3]
    POOLED[key] = torch.load(fp, map_location='cpu').float()  # (512,)
print('loaded image embeddings for', len(POOLED), 'volumes')
cache_domain = Counter('train' if k.startswith('train_') else
                       'hub_valid' if k.startswith('valid_') else 'unknown' for k in POOLED)
print('cache by CT-RATE Hub domain:', dict(cache_domain))
assert POOLED, 'No image embeddings found — run ctclip_features_colab.ipynb first.'
print('example dim:', tuple(next(iter(POOLED.values())).shape))

## 4. Frozen CT-CLIP text tower (prototypes + optional finding-name embeddings)

Weights stay frozen. Used only to build `PROTO[finding]` and (if `USE_LEARNED_FINDING_EMB=False`) a frozen finding-name vector for conditioning.

In [ ]:
from huggingface_hub import login, hf_hub_download
from ctclip_utils import CTCLIPEmbedder, REPO_ID, CTCLIP_WEIGHTS_HF
login()  # paste READ token if weights not already on Drive
os.makedirs(WEIGHTS, exist_ok=True)
wp = f'{WEIGHTS}/{os.path.basename(CTCLIP_WEIGHTS_HF)}'
if not os.path.exists(wp):
    wp = hf_hub_download(REPO_ID, CTCLIP_WEIGHTS_HF, repo_type='dataset', local_dir=WEIGHTS)
emb = CTCLIPEmbedder(wp)
print('CT-CLIP text tower ready on', emb.device)

## 5. Labels + strict Hub train / Hub validation separation

- Rows are `(pair, finding, y)` with `y in {worsened, stable, improved}`.
- Prefer tier=`explicit` (report_explicit).
- `train_*` -> train / tune (patient SHA partition). `valid_*` -> final test only.

In [ ]:
def vkey(v): return v.replace('.nii.gz', '').replace('.nii', '')

def volume_domain(v):
    k = vkey(v).lower()
    if k.startswith('train_'): return 'hub_train'
    if k.startswith('valid_'): return 'hub_valid'
    return 'unknown'

def pair_domain(pv, cv):
    a, b = volume_domain(pv), volume_domain(cv)
    return a if a == b else 'cross_domain'

def dev_partition(patient):
    raw = f'{SPLIT_SEED}|{patient}'.encode('utf-8')
    u = int.from_bytes(hashlib.sha256(raw).digest()[:8], 'big') / 2**64
    return 'tune' if u < TUNE_FRAC else 'train'

recs = {}
for line in open(LAB, encoding='utf-8'):
    if not line.strip(): continue
    x = json.loads(line)
    recs[(x['patient'], x['prior_volume'], x['curr_volume'])] = x

# optional real temporal / dynamic sentences (per pair)
dyn_of = {}
try:
    for line in open(LAB_DS, encoding='utf-8'):
        if not line.strip(): continue
        x = json.loads(line)
        ds = x.get('dynamic_sentences') or []
        if isinstance(ds, list): ds = ' '.join(s for s in ds if isinstance(s, str))
        dyn_of[(x['patient'], x['prior_volume'], x['curr_volume'])] = (ds or '').strip()
    print('loaded dynamic_sentences for', len(dyn_of), 'pairs')
except FileNotFoundError:
    print('WARN: dynamic-sentence file not found:', LAB_DS)

pair_ids = {}
examples = {'train': [], 'tune': [], 'test': []}
skipped = Counter()
candidate_pairs = Counter(); usable_pairs = Counter(); missing_hub_valid = []
for key, rec in recs.items():
    patient, pv, cv = key
    domain = pair_domain(pv, cv)
    if domain == 'hub_train':
        sp = dev_partition(patient)
    elif domain == 'hub_valid':
        sp = 'test'
    else:
        skipped[domain] += 1; continue
    if not rec.get('parse_ok'):
        skipped[f'no_label_{sp}'] += 1; continue
    candidate_pairs[sp] += 1
    if vkey(pv) not in POOLED or vkey(cv) not in POOLED:
        skipped[f'no_feature_{sp}'] += 1
        if sp == 'test': missing_hub_valid.append(key)
        continue
    usable_pairs[sp] += 1
    for fd in rec.get('findings', []):
        if fd.get('tier') != 'explicit':
            skipped['not_explicit'] += 1; continue
        d = fd.get('direction'); f = fd.get('finding')
        if d not in C2I or not f:
            skipped['bad_dir'] += 1; continue
        pid = pair_ids.setdefault(key, len(pair_ids))
        examples[sp].append({
            'vp': vkey(pv), 'vc': vkey(cv), 'patient': patient,
            'finding': f, 'hub_domain': domain,
            'y': C2I[d], 'pid': pid,
            'evidence': fd.get('evidence', '') or '',
            'dynamic': dyn_of.get(key, ''),
        })

for sp in ['train', 'tune']:
    assert all(e['hub_domain'] == 'hub_train' and e['vp'].startswith('train_') and
               e['vc'].startswith('train_') for e in examples[sp])
assert all(e['hub_domain'] == 'hub_valid' and e['vp'].startswith('valid_') and
           e['vc'].startswith('valid_') for e in examples['test'])
patients = {sp: {e['patient'] for e in examples[sp]} for sp in examples}
assert patients['train'].isdisjoint(patients['tune'])
assert patients['train'].isdisjoint(patients['test'])
assert patients['tune'].isdisjoint(patients['test'])
assert examples['train'] and examples['tune'] and examples['test']
if REQUIRE_COMPLETE_HUB_VALID_FEATURES:
    assert not missing_hub_valid, (
        f'{len(missing_hub_valid)} labeled Hub-validation pairs lack cached features; '
        f'finish encode_split("valid") first. First missing: {missing_hub_valid[:3]}')

print('candidate labeled pairs:', dict(candidate_pairs))
print('usable cached pairs    :', dict(usable_pairs))
for sp in ['train', 'tune', 'test']:
    cc = Counter(e['y'] for e in examples[sp])
    print(f'{sp:5}: {len(examples[sp]):5} ex / {len(patients[sp]):4} patients  '
          f'(worsened={cc[0]} stable={cc[1]} improved={cc[2]})')
print('skipped:', dict(skipped))
print('LEAKAGE CHECK PASSED: optimizer=train_* only; tune=train_* only; final test=valid_* only')

FINDINGS = ['Medical material','Arterial wall calcification','Cardiomegaly',
  'Pericardial effusion','Coronary artery wall calcification','Hiatal hernia',
  'Lymphadenopathy','Emphysema','Atelectasis','Lung nodule','Lung opacity',
  'Pulmonary fibrotic sequela','Pleural effusion','Mosaic attenuation pattern',
  'Peribronchial thickening','Consolidation','Bronchiectasis','Interlobular septal thickening']
F2I = {f: i for i, f in enumerate(FINDINGS)}
assert {e['finding'] for sp in examples for e in examples[sp]} <= set(FINDINGS)
print('canonical findings:', len(FINDINGS))
for sp in examples:
    for e in examples[sp]:
        e['fid'] = F2I[e['finding']]

## 6. Text prototypes (CE) + temporal sentences (contrastive)

**CE prototypes** — frozen 3-way bank per finding (`PROTO[f]`). Default = template bank. Set `PROTO_SOURCE='real'` to average train evince/dynamic text per (finding, class).

**Contrastive temporal text** — per example, prefer real evidence/dynamic sentences for worsened/improved/stable. If stable has no real sentence, assign **one fixed** synthetic template (seeded, not resampled each epoch). Embeddings cached with the frozen text tower.

In [ ]:
TEMPLATES = {
    'worsened': ['{f} has increased compared to the prior study',
                 '{f} has worsened since the previous exam',
                 'interval enlargement of {f}', 'new {f}', 'increased {f}'],
    'stable':   ['{f} is unchanged compared to the prior study',
                 'stable {f} with no interval change',
                 'no significant change in {f}', '{f} appears similar to prior'],
    'improved': ['{f} has decreased compared to the prior study',
                 '{f} has improved since the previous exam',
                 'interval decrease of {f}', '{f} has resolved', 'decreased {f}'],
}

# Configurable stable-only synthetic bank (used when no real temporal sentence exists)
STABLE_SYNTH_TEMPLATES = [
    'The {f} is unchanged from the prior examination.',
    'There has been no significant interval change in the {f}.',
    'The {f} remains stable compared with the prior examination.',
    'The {f} is stable compared with the previous study.',
]

def l2np(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

def embed_mean(texts):
    texts = [t for t in texts if t and str(t).strip()]
    if not texts:
        return None
    vecs = []
    B = 64
    for i in range(0, len(texts), B):
        vecs.append(emb.embed_texts(texts[i:i+B], normalize=True).numpy())
    e = np.concatenate(vecs, 0).mean(0)
    return l2np(e)

def embed_texts_batched(texts, bs=64):
    """Return (N,512) float32 L2-normalized embeddings."""
    out = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        out.append(emb.embed_texts(chunk, normalize=True).float().cpu())
    return torch.cat(out, 0) if out else torch.zeros(0, 512)

proto_cache = PROTO_PT if PROTO_SOURCE == 'templates' else PROTO_PT.replace('.pt', f'_{PROTO_SOURCE}.pt')

if os.path.exists(proto_cache) and PROTO_SOURCE == 'templates':
    PROTO = torch.load(proto_cache, map_location='cpu')
    print('loaded cached prototypes for', len(PROTO), 'findings from', proto_cache)
else:
    PROTO = {}
    from tqdm.auto import tqdm
    real_bank = defaultdict(lambda: defaultdict(list))
    if PROTO_SOURCE == 'real':
        for e in examples['train']:
            txt = (e.get('evidence') or e.get('dynamic') or '').strip()
            if txt:
                real_bank[e['finding']][e['y']].append(txt)
    for f in tqdm(FINDINGS):
        rows = []
        for ci, c in enumerate(CLASSES):
            if PROTO_SOURCE == 'real':
                vec = embed_mean(real_bank[f][ci])
                if vec is None:
                    prompts = [t.format(f=f.lower()) for t in TEMPLATES[c]]
                    vec = embed_mean(prompts)
            else:
                prompts = [t.format(f=f.lower()) for t in TEMPLATES[c]]
                vec = embed_mean(prompts)
            rows.append(vec)
        PROTO[f] = torch.tensor(np.stack(rows)).float()  # (3,512)
    torch.save(PROTO, proto_cache)
    print('built + cached prototypes for', len(PROTO), 'findings ->', proto_cache, '| source=', PROTO_SOURCE)

# optional frozen finding-name embeddings for conditioning (when not using learned emb)
FINDING_NAME_PT = f'{DRIVE}/ctclip_cache/finding_name_emb.pt'
if os.path.exists(FINDING_NAME_PT):
    FINDING_NAME_EMB = torch.load(FINDING_NAME_PT, map_location='cpu')
    print('loaded finding-name embeddings', FINDING_NAME_EMB.shape)
else:
    name_prompts = [f'finding: {f.lower()}' for f in FINDINGS]
    FINDING_NAME_EMB = emb.embed_texts(name_prompts, normalize=True).float().cpu()  # (18,512)
    torch.save(FINDING_NAME_EMB, FINDING_NAME_PT)
    print('built finding-name embeddings', FINDING_NAME_EMB.shape, '->', FINDING_NAME_PT)


# -------------------- per-example temporal sentence for contrastive --------------------
def _pick_real_temporal(e):
    """Prefer finding-level evidence, else pair-level dynamic."""
    ev = (e.get('evidence') or '').strip()
    if ev:
        return ev
    dyn = (e.get('dynamic') or '').strip()
    if dyn:
        return dyn
    return ''

def assign_temporal_texts(examples_dict, seed=STABLE_TEXT_SEED):
    """Attach temporal_text / temporal_src / synth_template_id. Stable synth fixed by seed."""
    rng = random.Random(seed)
    stats = Counter()
    synth_by_tid = Counter()
    for sp, exs in examples_dict.items():
        for i, e in enumerate(exs):
            real = _pick_real_temporal(e)
            y = e['y']
            if real:
                e['temporal_text'] = real
                e['temporal_src'] = 'real'
                e['synth_template_id'] = -1
                stats['real'] += 1
                stats[f'real_{I2C[y]}'] += 1
            elif y == C2I['stable']:
                tid = rng.randrange(len(STABLE_SYNTH_TEMPLATES))
                tmpl = STABLE_SYNTH_TEMPLATES[tid]
                e['temporal_text'] = tmpl.format(f=e['finding'].lower())
                e['temporal_src'] = 'synthetic'
                e['synth_template_id'] = tid
                stats['synthetic_stable'] += 1
                synth_by_tid[tid] += 1
            else:
                # worsened/improved with no real text: fall back to class template (last resort)
                # Spec prefers NOT replacing natural text; only used when extraction empty.
                cname = I2C[y]
                tmpl = TEMPLATES[cname][0]
                e['temporal_text'] = tmpl.format(f=e['finding'].lower())
                e['temporal_src'] = 'template_fallback'
                e['synth_template_id'] = -1
                stats['template_fallback'] += 1
                stats[f'fallback_{cname}'] += 1
    print('temporal text sources:', dict(stats))
    if synth_by_tid:
        print('synthetic stable template IDs:', dict(sorted(synth_by_tid.items())))
    return stats

_ = assign_temporal_texts(examples)

# Cache frozen 512-d embeddings for all temporal sentences (all splits)
def cache_temporal_text_embs(examples_dict, cache_path=TTEXT_PT):
    all_texts = []
    meta = []  # (sp, local_idx)
    for sp in ['train', 'tune', 'test']:
        for i, e in enumerate(examples_dict[sp]):
            all_texts.append(e['temporal_text'])
            meta.append((sp, i))
    # unique strings to avoid re-encoding duplicates
    uniq = {}
    ordered_unique = []
    for t in all_texts:
        if t not in uniq:
            uniq[t] = len(ordered_unique)
            ordered_unique.append(t)
    print(f'encoding {len(ordered_unique)} unique temporal sentences '
          f'(of {len(all_texts)} examples) ...')
    from tqdm.auto import tqdm
    uniq_emb = embed_texts_batched(ordered_unique, bs=64)
    # map back
    by_sp = {sp: [None] * len(examples_dict[sp]) for sp in examples_dict}
    for (sp, i), t in zip(meta, all_texts):
        by_sp[sp][i] = uniq_emb[uniq[t]]
    out = {sp: torch.stack(by_sp[sp], 0) for sp in by_sp}  # (N,512)
    payload = {
        'emb': out,
        'texts': {sp: [e['temporal_text'] for e in examples_dict[sp]] for sp in examples_dict},
        'src': {sp: [e['temporal_src'] for e in examples_dict[sp]] for sp in examples_dict},
        'synth_id': {sp: [e['synth_template_id'] for e in examples_dict[sp]] for sp in examples_dict},
    }
    torch.save(payload, cache_path)
    print('cached temporal text emb ->', cache_path,
          {sp: tuple(out[sp].shape) for sp in out})
    return out

# Always rebuild when examples change (hash of first/last texts + n)
_tkey = (
    len(examples['train']), len(examples['tune']), len(examples['test']),
    examples['train'][0]['temporal_text'][:80] if examples['train'] else '',
    STABLE_TEXT_SEED,
)
if os.path.exists(TTEXT_PT):
    _prev = torch.load(TTEXT_PT, map_location='cpu')
    _ok = (
        isinstance(_prev, dict) and 'emb' in _prev
        and all(sp in _prev['emb'] for sp in ['train', 'tune', 'test'])
        and all(_prev['emb'][sp].shape[0] == len(examples[sp]) for sp in ['train', 'tune', 'test'])
    )
else:
    _ok = False

if _ok:
    TEMPORAL_TEXT_EMB = {sp: _prev['emb'][sp].float() for sp in ['train', 'tune', 'test']}
    print('loaded temporal text emb cache', {sp: tuple(TEMPORAL_TEXT_EMB[sp].shape) for sp in TEMPORAL_TEXT_EMB})
else:
    TEMPORAL_TEXT_EMB = cache_temporal_text_embs(examples)

# quick stable-source summary on train
_src_tr = Counter(e['temporal_src'] for e in examples['train'])
print('train temporal_src:', dict(_src_tr))

## 7. Tensorize splits

In [ ]:
def tensorize(exs, ttext_emb):
    VP = torch.stack([POOLED[e['vp']] for e in exs])
    VC = torch.stack([POOLED[e['vc']] for e in exs])
    PR = torch.stack([PROTO[e['finding']] for e in exs])          # (N,3,512)
    Y  = torch.tensor([e['y'] for e in exs], dtype=torch.long)
    FID = torch.tensor([e['fid'] for e in exs], dtype=torch.long)
    PID = torch.tensor([e['pid'] for e in exs], dtype=torch.long)
    TT = ttext_emb.float().clone()                                 # (N,512) frozen temporal sentence emb
    SRC = [e['temporal_src'] for e in exs]
    TID = torch.tensor([e['synth_template_id'] for e in exs], dtype=torch.long)
    Fn = [e['finding'] for e in exs]
    Ttxt = [e['temporal_text'] for e in exs]
    return {
        'vp': VP, 'vc': VC, 'pr': PR, 'y': Y, 'fid': FID, 'pid': PID,
        'tt': TT, 'src': SRC, 'synth_id': TID, 'fn': Fn, 'ttxt': Ttxt,
    }

DATA = {sp: tensorize(examples[sp], TEMPORAL_TEXT_EMB[sp]) for sp in ['train', 'tune', 'test']}
cnt = Counter(DATA['train']['y'].tolist()); tot = sum(cnt.values())
W_cls = torch.tensor([tot / (3 * max(cnt[i], 1)) for i in range(3)], dtype=torch.float32)
print('class weights (w/s/i):', [round(x, 3) for x in W_cls.tolist()])
for sp in DATA:
    shapes = {k: (tuple(v.shape) if torch.is_tensor(v) else len(v)) for k, v in DATA[sp].items()}
    print(sp, shapes)
print('train tt emb finite:', torch.isfinite(DATA['train']['tt']).all().item())

## 8. Trainable module — finding-conditioned Difference Transformer

Only this module is trained. CT-CLIP stays frozen.

- Inputs: `v_prior`, `v_current`, `finding_id`
- Conditioning: `e_diff <- e_diff + e_f` (default) or finding as 4th token
- Output: `d_f` (512-d change emb for **this** finding on **this** pair) + optional `mag`
-     }

DATA = {sp: tensorize(examples[sp], TEMPORAL_TEXT_EMB(same-finding masked SupCon)

In [ ]:
class DifferenceTransformer(nn.Module):
    """Finding-conditioned temporal difference module. d_f = g(vp, vc, finding)."""

    def __init__(self, n_findings=18, d_in=512, d_model=256, n_layers=2, n_heads=4,
                 dropout=0.1, antisym=False, magnitude=False,
                 finding_conditioning=True, finding_as_4th_token=False,
                 use_learned_finding_emb=True, frozen_finding_emb=None,
                 tau_con_init=0.07, learnable_tau_con=True):
        super().__init__()
        self.finding_conditioning = finding_conditioning
        self.finding_as_4th_token = finding_as_4th_token and finding_conditioning
        self.antisym = antisym
        self.W = nn.Linear(d_in, d_model)
        self.role = nn.Parameter(torch.randn(2, d_model) * 0.02)
        self.e_diff = nn.Parameter(torch.randn(1, d_model) * 0.02)
        if finding_conditioning:
            if use_learned_finding_emb:
                self.finding_emb = nn.Embedding(n_findings, d_model)
                nn.init.normal_(self.finding_emb.weight, std=0.02)
            else:
                assert frozen_finding_emb is not None
                self.register_buffer('finding_name_512', frozen_finding_emb.float())
                self.finding_proj = nn.Linear(d_in, d_model)
                self.finding_emb = None
        else:
            self.finding_emb = None
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * 4, dropout=dropout,
            batch_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, d_in)
        self.mag_head = nn.Linear(d_model, 1) if magnitude else None
        self.logit_scale = nn.Parameter(torch.tensor(float(np.log(1 / 0.07))))
        log_tau = float(np.log(tau_con_init))
        if learnable_tau_con:
            self.log_tau_con = nn.Parameter(torch.tensor(log_tau))
        else:
            self.register_buffer('log_tau_con', torch.tensor(log_tau))

    def _e_f(self, fid):
        if not self.finding_conditioning:
            return None
        if self.finding_emb is not None:
            return self.finding_emb(fid)
        return self.finding_proj(self.finding_name_512[fid])

    def _pass(self, vp, vc, fid):
        B = vp.size(0)
        tp = self.W(vp) + self.role[0]
        tc = self.W(vc) + self.role[1]
        ed = self.e_diff.expand(B, -1).clone()
        ef = self._e_f(fid)
        if ef is not None and not self.finding_as_4th_token:
            ed = ed + ef
            seq = torch.stack([ed, tp, tc], dim=1)
        elif ef is not None and self.finding_as_4th_token:
            seq = torch.stack([ed, tp, tc, ef], dim=1)
        else:
            seq = torch.stack([ed, tp, tc], dim=1)
        h = self.enc(seq)
        hdiff = h[:, 0]
        mag = self.mag_head(hdiff).squeeze(-1) if self.mag_head is not None else None
        return self.head(hdiff), mag

    def forward(self, vp, vc, fid=None):
        if fid is None:
            fid = torch.zeros(vp.size(0), dtype=torch.long, device=vp.device)
        vd, mag = self._pass(vp, vc, fid)
        if self.antisym:
            vd_rev, _ = self._pass(vc, vp, fid)
            vd = vd - vd_rev
        return vd, mag

    def tau_con(self):
        return self.log_tau_con.exp().clamp(min=1e-3, max=1.0)


def logits_from(vd, proto, logit_scale):
    vd = F.normalize(vd, dim=-1)
    pr = F.normalize(proto, dim=-1)
    cos = torch.einsum('bd,bkd->bk', vd, pr)
    return logit_scale.exp().clamp(max=100) * cos


def _masked_crossmodal_one_way(anchor, other, y_a, y_o, fid_a, fid_o, tau):
    """SupCon average-over-positives: anchor retrieves among other.

    Positives: same finding AND same label.
    Negatives: same finding AND different label.
    Ignored: different finding (not in denominator).
    Valid anchor: >=1 positive AND >=1 same-finding different-label negative.
    """
    anchor = F.normalize(anchor, dim=-1)
    other = F.normalize(other, dim=-1)
    B, M = anchor.size(0), other.size(0)
    if B == 0 or M == 0:
        return anchor.new_zeros(()), {'n_valid': 0}

    sim = (anchor @ other.t()) / tau

    same_f = fid_a.unsqueeze(1).eq(fid_o.unsqueeze(0))
    same_y = y_a.unsqueeze(1).eq(y_o.unsqueeze(0))
    pos_mask = same_f & same_y
    neg_mask = same_f & ~same_y
    allowed = same_f

    pos_counts = pos_mask.sum(dim=1).float()
    neg_counts = neg_mask.sum(dim=1).float()
    valid = (pos_counts >= 1) & (neg_counts >= 1)
    n_valid = int(valid.sum().item())
    if n_valid == 0:
        return anchor.new_zeros(()), {
            'n_valid': 0, 'n_anchors': B,
            'mean_pos': 0.0, 'mean_neg': 0.0,
        }

    neg_large = torch.finfo(sim.dtype).min / 4
    logits = sim.masked_fill(~allowed, neg_large)
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    exp_logits = logits.exp() * allowed.float()
    log_denom = exp_logits.sum(dim=1, keepdim=True).clamp(min=1e-8).log()
    log_prob = logits - log_denom

    pos_log = torch.where(pos_mask, log_prob, torch.zeros_like(log_prob))
    mean_pos = pos_log.sum(dim=1) / pos_counts.clamp(min=1.0)
    loss = -mean_pos[valid].mean()
    if not torch.isfinite(loss):
        loss = anchor.new_zeros(())

    with torch.no_grad():
        cos = anchor @ other.t()

        def _row_mean(x, mask):
            s = torch.where(mask, x, torch.zeros_like(x)).sum(dim=1)
            c = mask.sum(dim=1).clamp(min=1).float()
            return (s / c)[valid].mean().item() if valid.any() else 0.0

        stats = {
            'n_valid': n_valid,
            'n_anchors': B,
            'mean_pos_count': float(pos_counts[valid].mean().item()),
            'mean_neg_count': float(neg_counts[valid].mean().item()),
            'mean_pos_cos': _row_mean(cos, pos_mask),
            'mean_neg_cos': _row_mean(cos, neg_mask),
        }
    return loss, stats


def masked_crossmodal_supcon_loss(d, t, y, fid, tau, symmetric=False):
    """Cross-modal masked SupCon: d (image temporal) vs t (frozen temporal text)."""
    t = t.detach()
    loss_i2t, st = _masked_crossmodal_one_way(d, t, y, y, fid, fid, tau)
    if not symmetric:
        return loss_i2t, st
    loss_t2i, st2 = _masked_crossmodal_one_way(t, d, y, y, fid, fid, tau)
    st = {**st, 'n_valid_t2i': st2.get('n_valid', 0)}
    return 0.5 * (loss_i2t + loss_t2i), st


def masked_supcon_loss(z, y, fid, tau):
    """DEPRECATED embedding-only SupCon. Prefer masked_crossmodal_supcon_loss."""
    z = F.normalize(z, dim=-1)
    B = z.size(0)
    if B < 2:
        return z.new_zeros(())
    sim = (z @ z.t()) / tau
    self_mask = torch.eye(B, dtype=torch.bool, device=z.device)
    same_f = fid.unsqueeze(0).eq(fid.unsqueeze(1)) & ~self_mask
    pos_mask = same_f & y.unsqueeze(0).eq(y.unsqueeze(1))
    allowed = same_f
    pos_counts = pos_mask.sum(dim=1).float()
    neg_counts = (same_f & ~y.unsqueeze(0).eq(y.unsqueeze(1))).sum(dim=1).float()
    valid = (pos_counts > 0) & (neg_counts > 0)
    if not valid.any():
        return z.new_zeros(())
    neg_large = torch.finfo(sim.dtype).min / 2
    logits = sim.masked_fill(self_mask | ~allowed, neg_large)
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    exp_logits = logits.exp() * allowed.float()
    log_prob = logits - exp_logits.sum(dim=1, keepdim=True).clamp(min=1e-8).log()
    pos_log = torch.where(pos_mask, log_prob, torch.zeros_like(log_prob))
    mean_pos = pos_log.sum(dim=1) / pos_counts.clamp(min=1.0)
    loss = -mean_pos[valid].mean()
    return loss if torch.isfinite(loss) else z.new_zeros(())


npar = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
_probe = DifferenceTransformer(
    n_findings=len(FINDINGS), d_model=D_MODEL, antisym=ANTISYM, magnitude=USE_MAGNITUDE,
    finding_conditioning=FINDING_CONDITIONING, finding_as_4th_token=FINDING_AS_4TH_TOKEN,
    use_learned_finding_emb=USE_LEARNED_FINDING_EMB,
    frozen_finding_emb=None if USE_LEARNED_FINDING_EMB else FINDING_NAME_EMB,
    tau_con_init=TAU_CON_INIT, learnable_tau_con=LEARNABLE_TAU_CON,
)
print('module trainable params:', f'{npar(_probe):,}')
del _probe

## 9. Contrastive-aware batch sampler

Build batches that cover ~`K_FINDINGS_PER_BATCH` findings with rows from multiple direction classes so masked cross-modal SupCon has same-finding positives **and** other-direction negatives among temporal texts.

In [ ]:
def build_buckets(split):
    """Map (fid, y) -> list of dataset indices for contrastive-aware sampling."""
    y = DATA[split]['y']; fid = DATA[split]['fid']
    buckets = defaultdict(list)
    for i in range(len(y)):
        buckets[(int(fid[i]), int(y[i]))].append(i)
    return buckets

TRAIN_BUCKETS = build_buckets('train')
print('train buckets (fid,y) with data:', len(TRAIN_BUCKETS),
      '| findings present:', len({k[0] for k in TRAIN_BUCKETS}))

def sample_contrastive_batch(buckets, k_findings=K_FINDINGS_PER_BATCH,
                             n_per_class=N_PER_CLASS, max_bs=MAX_BATCH_SIZE, rng=None):
    rng = rng or random
    by_f = defaultdict(set)
    for (f, y), idxs in buckets.items():
        if idxs:
            by_f[f].add(y)
    eligible = [f for f, ys in by_f.items() if len(ys) >= 2]
    if not eligible:
        eligible = list(by_f.keys())
    if not eligible:
        return []
    k = min(k_findings, len(eligible))
    chosen_f = rng.sample(eligible, k)
    batch = []
    for f in chosen_f:
        for y in range(3):
            pool = buckets.get((f, y), [])
            if not pool:
                continue
            take = min(n_per_class, len(pool))
            batch.extend(rng.sample(pool, take) if len(pool) >= take else list(pool))
    if not batch:
        return []
    if len(batch) > max_bs:
        batch = rng.sample(batch, max_bs)
    rng.shuffle(batch)
    return batch

def steps_per_epoch(n, approx_bs):
    return max(1, math.ceil(n / max(approx_bs, 1)))

_bs_probe = [len(sample_contrastive_batch(TRAIN_BUCKETS)) for _ in range(20)]
print('probe batch sizes: min/mean/max =', min(_bs_probe),
      round(sum(_bs_probe)/len(_bs_probe), 1), max(_bs_probe))
APPROX_BS = max(int(sum(_bs_probe)/len(_bs_probe)), 32)
STEPS = steps_per_epoch(len(DATA['train']['y']), APPROX_BS)
print('steps/epoch ~', STEPS, '| approx_bs ~', APPROX_BS)

## 10. Train on Hub `train_*` (early-stop on patient-held-out `train_*` tune)

Total loss:

**L = λ_ce L_CE + λ_mag L_mag + λ_con L_con**

- `L_CE` — weighted CE on `cosine(d_f, PROTO[f]) * exp(logit_scale)`
- `L_mag` — optional BCE(mag, change vs stable)
- `L_con` — masked same-finding **cross-modal** SupCon: `d_f` ↔ frozen temporal-sentence emb `t` (temperature `tau_con`)
  - positives: same finding + same direction texts (incl. own sentence)
  - negatives: same finding + other direction
  - ignored: other findings
  - optional symmetric text→d when SYMMETRIC=True`

In [ ]:
model = DifferenceTransformer(
    n_findings=len(FINDINGS),
    d_model=D_MODEL,
    antisym=ANTISYM,
    magnitude=USE_MAGNITUDE,
    finding_conditioning=FINDING_CONDITIONING,
    finding_as_4th_token=FINDING_AS_4TH_TOKEN,
    use_learned_finding_emb=USE_LEARNED_FINDING_EMB,
    frozen_finding_emb=None if USE_LEARNED_FINDING_EMB else FINDING_NAME_EMB,
    tau_con_init=TAU_CON_INIT,
    learnable_tau_con=LEARNABLE_TAU_CON,
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ce_loss_fn = nn.CrossEntropyLoss(weight=W_cls.to(DEVICE))


@torch.no_grad()
def evaluate(split):
    model.eval()
    D = DATA[split]
    all_y, all_p = [], []
    tot, n = 0.0, 0
    bs = 512
    for i in range(0, len(D['y']), bs):
        sl = slice(i, i + bs)
        vp = D['vp'][sl].to(DEVICE)
        vc = D['vc'][sl].to(DEVICE)
        pr = D['pr'][sl].to(DEVICE)
        y = D['y'][sl].to(DEVICE)
        fid = D['fid'][sl].to(DEVICE)
        vd, mag = model(vp, vc, fid)
        lg = logits_from(vd, pr, model.logit_scale)
        loss = ce_loss_fn(lg, y)
        tot += loss.item() * len(y)
        n += len(y)
        all_y += y.cpu().tolist()
        all_p += lg.argmax(1).cpu().tolist()
    mf1 = f1_score(all_y, all_p, labels=[0, 1, 2], average='macro', zero_division=0)
    return tot / max(n, 1), mf1, np.array(all_y), np.array(all_p)


def train_epoch():
    model.train()
    tot, n = 0.0, 0
    sum_ce = sum_mag = sum_con = 0.0
    n_con_valid = n_con_batches = n_zero_valid = 0
    all_y, all_p = [], []
    last_st = {}
    for _ in range(STEPS):
        if USE_SUPCON:
            idxs = sample_contrastive_batch(TRAIN_BUCKETS)
            if len(idxs) < 4:
                n_train = len(DATA['train']['y'])
                idxs = random.sample(range(n_train), min(MAX_BATCH_SIZE, n_train))
        else:
            n_train = len(DATA['train']['y'])
            idxs = random.sample(range(n_train), min(MAX_BATCH_SIZE, n_train))
        idxs_t = torch.tensor(idxs, dtype=torch.long)
        vp = DATA['train']['vp'][idxs_t].to(DEVICE)
        vc = DATA['train']['vc'][idxs_t].to(DEVICE)
        pr = DATA['train']['pr'][idxs_t].to(DEVICE)
        y = DATA['train']['y'][idxs_t].to(DEVICE)
        fid = DATA['train']['fid'][idxs_t].to(DEVICE)
        tt = DATA['train']['tt'][idxs_t].to(DEVICE)

        vd, mag = model(vp, vc, fid)
        lg = logits_from(vd, pr, model.logit_scale)

        loss = vd.new_zeros(())
        l_ce = vd.new_zeros(())
        l_mag = vd.new_zeros(())
        l_con = vd.new_zeros(())
        if USE_CE:
            l_ce = ce_loss_fn(lg, y)
            loss = loss + LAMBDA_CE * l_ce
        if USE_MAGNITUDE and mag is not None:
            is_change = (y != C2I['stable']).float()
            l_mag = F.binary_cross_entropy_with_logits(mag, is_change)
            loss = loss + LAMBDA_MAG * l_mag
        if USE_SUPCON:
            l_con, st = masked_crossmodal_supcon_loss(
                vd, tt, y, fid, model.tau_con(),
                symmetric=CONTRASTIVE_SYMMETRIC)
            loss = loss + LAMBDA_CON * l_con
            last_st = st
            n_con_batches += 1
            n_con_valid += st.get('n_valid', 0)
            if st.get('n_valid', 0) == 0:
                n_zero_valid += 1

        if not torch.isfinite(loss) or (
            loss.item() == 0.0 and not (USE_CE or USE_MAGNITUDE or USE_SUPCON)
        ):
            loss = vd.sum() * 0.0

        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        bs = len(idxs)
        tot += loss.item() * bs
        n += bs
        sum_ce += float(l_ce.item()) * bs
        sum_mag += float(l_mag.item()) * bs
        sum_con += float(l_con.item()) * bs
        all_y += y.cpu().tolist()
        all_p += lg.argmax(1).cpu().tolist()
    mf1 = f1_score(all_y, all_p, labels=[0, 1, 2], average='macro', zero_division=0)
    return dict(
        loss=tot / max(n, 1),
        ce=sum_ce / max(n, 1),
        mag=sum_mag / max(n, 1),
        con=sum_con / max(n, 1),
        mf1=mf1,
        mean_valid_per_batch=n_con_valid / max(n_con_batches, 1),
        zero_valid_batches=n_zero_valid,
        last_st=last_st,
    )


best, best_state, bad = -1.0, None, 0
for ep in range(1, EPOCHS + 1):
    tr = train_epoch()
    _, vf1, _, _ = evaluate('tune')
    if vf1 > best:
        best = vf1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if ep % 10 == 0 or ep == 1:
        st = tr.get('last_st') or {}
        msg = (
            'ep %3d  loss %.3f  (ce=%.3f mag=%.3f con=%.3f)  '
            'tune_macroF1 %.3f  (best %.3f)  tau_con=%.4f  logit_scale=%.2f  '
            'con_valid~%.1f  pos_cos=%.3f  neg_cos=%.3f'
        ) % (
            ep, tr['loss'], tr['ce'], tr['mag'], tr['con'],
            vf1, best, model.tau_con().item(), model.logit_scale.exp().item(),
            tr['mean_valid_per_batch'],
            st.get('mean_pos_cos', float('nan')),
            st.get('mean_neg_cos', float('nan')),
        )
        print(msg)
        if tr['zero_valid_batches']:
            print('  warn: %d batches with 0 valid contrastive anchors' % tr['zero_valid_batches'])
    if bad >= PATIENCE:
        print('early stop @ ep %d  best tune macro-F1 %.3f' % (ep, best))
        break

assert best_state is not None
model.load_state_dict(best_state)
print('restored best. train-domain tune macro-F1 =', round(best, 3))
print('flags:', dict(
    FINDING_CONDITIONING=FINDING_CONDITIONING,
    FINDING_AS_4TH_TOKEN=FINDING_AS_4TH_TOKEN,
    USE_CE=USE_CE, USE_MAGNITUDE=USE_MAGNITUDE, USE_SUPCON=USE_SUPCON,
    CONTRASTIVE_SYMMETRIC=CONTRASTIVE_SYMMETRIC,
    PROTO_SOURCE=PROTO_SOURCE, ANTISYM=ANTISYM,
    LAMBDA_CE=LAMBDA_CE, LAMBDA_MAG=LAMBDA_MAG, LAMBDA_CON=LAMBDA_CON))

## 11. FINAL TEST — CT-RATE Hub `valid_*` only

Run **once** after architecture / hyperparameters are fixed. Do not tune from these numbers.

Inference: `d_f = g(v_p, v_c, f)` -> `argmax cos(d_f, PROTO[f])` — **no report text**.

In [ ]:
_, macro, y, pred = evaluate('test')
percls = f1_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
acc = (y == pred).mean()
maj = Counter(DATA['train']['y'].tolist()).most_common(1)[0][0]
maj_macro = f1_score(y, np.full_like(y, maj), labels=[0, 1, 2], average='macro', zero_division=0)
assert all(e['hub_domain'] == 'hub_valid' for e in examples['test'])
print('=== FINAL HUB-VALIDATION TEST — per class ===')
print(f'accuracy : {acc:.3f}')
print(f'macro-F1 : {macro:.3f}   (always-{CLASSES[maj]} ref = {maj_macro:.3f})')
for i, c in enumerate(CLASSES):
    print(f'  F1 {c:9}: {percls[i]:.3f}')
print('\nconfusion (rows=true, cols=pred; order w/s/i):')
print(confusion_matrix(y, pred, labels=[0, 1, 2]))

## 12. FINAL HUB-VALIDATION TEST — per disease (finding)

In [ ]:
model.eval()
D = DATA['test']
all_p = []
with torch.no_grad():
    bs = 512
    for i in range(0, len(D['y']), bs):
        sl = slice(i, i + bs)
        vd, _ = model(D['vp'][sl].to(DEVICE), D['vc'][sl].to(DEVICE), D['fid'][sl].to(DEVICE))
        lg = logits_from(vd, D['pr'][sl].to(DEVICE), model.logit_scale)
        all_p.append(lg.argmax(1).cpu())
pred = torch.cat(all_p).numpy()
y = D['y'].numpy()
Fn = D['fn']
by_f = defaultdict(lambda: {'y': [], 'p': []})
for yi, pi, fi in zip(y, pred, Fn):
    by_f[fi]['y'].append(yi); by_f[fi]['p'].append(pi)
rows = []
for f, d in by_f.items():
    yy, pp = np.array(d['y']), np.array(d['p'])
    present = sorted(set(yy.tolist()))
    rows.append((f, len(yy), (yy == pp).mean(),
                 f1_score(yy, pp, labels=present, average='macro', zero_division=0)))
rows.sort(key=lambda r: -r[1])
print('=== FINAL HUB-VALIDATION TEST — per disease ===')
print(f"{'finding':<34}{'n':>5}{'acc':>7}{'macroF1*':>10}")
for f, n, a, mf1 in rows:
    print(f'{f:<34}{n:>5}{a:>7.3f}{mf1:>10.3f}')
print('\n* macro-F1 over classes actually present for that finding')

## 13. Save checkpoint

In [ ]:
ckpt_dir = f'{DRIVE}/ctclip_cache/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
tag = (
    f"fc{int(FINDING_CONDITIONING)}_mag{int(USE_MAGNITUDE)}_supcon{int(USE_SUPCON)}"
    f"_sym{int(CONTRASTIVE_SYMMETRIC)}_proto{PROTO_SOURCE}_seed{SPLIT_SEED}"
)
ckpt_path = f'{ckpt_dir}/proposed_{tag}.pt'
torch.save({
    'model': best_state,
    'tune_macro_f1': best,
    'findings': FINDINGS,
    'classes': CLASSES,
    'config': dict(
        FINDING_CONDITIONING=FINDING_CONDITIONING,
        FINDING_AS_4TH_TOKEN=FINDING_AS_4TH_TOKEN,
        USE_LEARNED_FINDING_EMB=USE_LEARNED_FINDING_EMB,
        USE_CE=USE_CE,
        USE_MAGNITUDE=USE_MAGNITUDE,
        USE_SUPCON=USE_SUPCON,
        CONTRASTIVE_SYMMETRIC=CONTRASTIVE_SYMMETRIC,
        LEARNABLE_TAU_CON=LEARNABLE_TAU_CON,
        PROTO_SOURCE=PROTO_SOURCE,
        ANTISYM=ANTISYM,
        D_MODEL=D_MODEL, LR=LR,
        LAMBDA_CE=LAMBDA_CE, LAMBDA_MAG=LAMBDA_MAG, LAMBDA_CON=LAMBDA_CON,
        TAU_CON_INIT=TAU_CON_INIT,
        STABLE_TEXT_SEED=STABLE_TEXT_SEED,
        K_FINDINGS_PER_BATCH=K_FINDINGS_PER_BATCH,
        N_PER_CLASS=N_PER_CLASS,
    ),
}, ckpt_path)
print('saved', ckpt_path)

## Notes / ablations

| Knob | Suggested sweeps |
|---|---|
| `FINDING_CONDITIONING` | `True` (proposed) vs `False` (current shared-`d` baseline) |
| `FINDING_AS_4TH_TOKEN` | `False` (`e_diff+e_f`) vs `True` (4th token) |
| `USE_CE` / `USE_MAGNITUDE` / `USE_SUPCON` | all on/off combos (keep CE on for classification) |
| `CONTRASTIVE_SYMMETRIC` | `False` (d→text) vs `True` (bidirectional) |
| `LAMBDA_CON` | `{0.1, 0.25        CONTRASO_SOURCE` | `templates` vs `real` (CE prototypes only) |
| `USE_LEARNED_FINDING_EMB` | learned vs frozen text-name emb |

**Cross-modal contrastive design**
- `d_f` aligns to **frozen temporal-sentence embeddings** (evidence/dynamic; synthetic stable templates only when needed).
- Same-finding mask: + same direction texts, − other direction, ignore other findings.
- SupCon averages log-prob over **each** positive (not log-sum of positives).
- Valid anchor needs ≥1 positive **and** ≥1 same-finding negative.
- `tau_con` is **separate** from CE `logit_scale`.
- Magnitude is aux only; inference is still pure prototype argmax (**no report text** at test).

**Protocol (do not break)**
- Optimize + early-stop on Hub `train_*` only (`tune` = patient-held-out fraction).
- Inspect Hub `valid_*` **once** after freezing choices.

See `docs/proposed_pipeline.png` and `docs/current_pipeline.png`.

## 14. Diagnostics — stable templates vs CE prototypes + contrastive mask sanity

Encode the synthetic stable bank and compare to each finding's CE prototypes. Then run a tiny synthetic batch that prints positive / negative / ignored masks for the cross-modal loss.

In [ ]:
# ---- stable template vs PROTO diagnostic ----
print('=== Stable synthetic templates vs CE prototypes ===')
rows = []
for f in FINDINGS:
    pr = F.normalize(PROTO[f].float(), dim=-1)
    for tid, tmpl in enumerate(STABLE_SYNTH_TEMPLATES):
        sent = tmpl.format(f=f.lower())
        te = emb.embed_texts([sent], normalize=True).float().cpu()[0]
        te = F.normalize(te, dim=-1)
        cos = (pr @ te).tolist()
        flag = ''
        if cos[1] < cos[0] or cos[1] < cos[2]:
            flag = 'WARN: stable tmpl closer to change class than stable proto'
        rows.append(dict(
            finding=f, tid=tid,
            cos_w=cos[0], cos_s=cos[1], cos_i=cos[2],
            flag=flag, text=sent,
        ))

f0 = FINDINGS[0]
tvecs = emb.embed_texts(
    [t.format(f=f0.lower()) for t in STABLE_SYNTH_TEMPLATES],
    normalize=True,
).float().cpu()
tvecs = F.normalize(tvecs, dim=-1)
pair = (tvecs @ tvecs.t()).numpy()
print('pairwise cos among stable templates (finding=%r):' % (f0,))
print(np.round(pair, 3))
warn = [r for r in rows if r['flag']]
print('template-proto rows: %d | warnings: %d' % (len(rows), len(warn)))
for r in warn[:5]:
    print(' ', r['finding'], 'tid', r['tid'],
          'cos_w/s/i=%.3f/%.3f/%.3f' % (r['cos_w'], r['cos_s'], r['cos_i']),
          r['flag'])

_diag_path = f'{DRIVE}/ctclip_cache/stable_template_proto_diag.csv'
try:
    import csv as _csv
    with open(_diag_path, 'w', newline='', encoding='utf-8') as f:
        w = _csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print('wrote', _diag_path)
except Exception as e:
    print('diag csv skip:', e)

# ---- synthetic minibatch mask demo (2 findings x 3 classes) ----
print('')
print('=== Synthetic mask demo (2 findings x 3 classes) ===')
_fid = torch.tensor([0, 0, 0, 1, 1, 1])
_y = torch.tensor([0, 1, 2, 0, 1, 2])
_d = F.normalize(torch.randn(6, 512), dim=-1)
_t = F.normalize(_d + 0.1 * torch.randn_like(_d), dim=-1)

same_f = _fid.unsqueeze(1).eq(_fid.unsqueeze(0))
same_y = _y.unsqueeze(1).eq(_y.unsqueeze(0))
pos_m = same_f & same_y
neg_m = same_f & ~same_y
ign_m = ~same_f
print('fid:', _fid.tolist())
print('y  :', _y.tolist())
print('POS mask:')
print(pos_m.int().numpy())
print('NEG mask:')
print(neg_m.int().numpy())
print('IGN mask:')
print(ign_m.int().numpy())

assert pos_m[0].tolist() == [True, False, False, False, False, False]
assert neg_m[0].tolist() == [False, True, True, False, False, False]
assert ign_m[0].tolist() == [False, False, False, True, True, True]
assert pos_m[0, 0] and pos_m[3, 3]

loss_demo, st_demo = masked_crossmodal_supcon_loss(
    _d, _t, _y, _fid, torch.tensor(0.07), symmetric=False)
print('demo loss', float(loss_demo), 'stats', st_demo)
assert st_demo['n_valid'] == 6
print('MASK SANITY OK')

## 15. Loss ablation sweep — CE × Mag × SupCon

Runs all loss combinations with a **fresh model** each time. Early-stops on **tune** macro-F1; scores Hub `valid_*` **once** per combo.

| Flag | Meaning |
|------|---------|
| CE | Prototype cross-entropy (classification readout) |
| Mag | Magnitude BCE (change vs stable) |
| SupCon | Masked **cross-modal** SupCon: `d_f` ↔ frozen temporal-sentence emb |

Defaults: skip `(0,0,0)`; set `QUICK_ABLATION=True` for CE-on only (4 runs).  
Checkpoints + CSV → `Drive/.../ctclip_cache/ablations/`.

**Requires:** `DATA[*]['tt']` and `masked_crossmodal_supcon_loss` (cells above).

In [ ]:
# Loss ablation: CE x Mag x SupCon (cross-modal d <-> temporal text)
import os, random, time
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

SKIP_ALL_OFF = True
QUICK_ABLATION = False
ABLATION_EPOCHS = EPOCHS
ABLATION_PATIENCE = PATIENCE
ABLATION_SEED = 0
RUN_TEST_EACH = True
SAVE_CKPTS = True
SAVE_CSV = True

_SYM = globals().get('CONTRASTIVE_SYMMETRIC', False)
_LEARN_TAU = globals().get('LEARNABLE_TAU_CON', True)

ABL_DIR = f'{DRIVE}/ctclip_cache/ablations'
os.makedirs(ABL_DIR, exist_ok=True)
ts = time.strftime('%Y%m%d_%H%M%S')

assert 'tt' in DATA['train'], "DATA['train']['tt'] missing — run temporal-text + tensorize cells first"
assert 'masked_crossmodal_supcon_loss' in globals(), 'masked_crossmodal_supcon_loss missing — run model cell'

def _set_seed(s):
    torch.manual_seed(s)
    np.random.seed(s)
    random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def _make_model():
    kwargs = dict(
        n_findings=len(FINDINGS),
        d_model=D_MODEL,
        antisym=ANTISYM,
        magnitude=True,
        finding_conditioning=FINDING_CONDITIONING,
        finding_as_4th_token=FINDING_AS_4TH_TOKEN,
        use_learned_finding_emb=USE_LEARNED_FINDING_EMB,
        frozen_finding_emb=None if USE_LEARNED_FINDING_EMB else FINDING_NAME_EMB,
        tau_con_init=TAU_CON_INIT,
    )
    try:
        return DifferenceTransformer(**kwargs, learnable_tau_con=_LEARN_TAU).to(DEVICE)
    except TypeError:
        return DifferenceTransformer(**kwargs).to(DEVICE)

@torch.no_grad()
def _evaluate(model, split):
    model.eval()
    D = DATA[split]
    ce_fn = nn.CrossEntropyLoss(weight=W_cls.to(DEVICE))
    all_y, all_p = [], []
    tot, n = 0.0, 0
    bs = 512
    for i in range(0, len(D['y']), bs):
        sl = slice(i, i + bs)
        vp = D['vp'][sl].to(DEVICE)
        vc = D['vc'][sl].to(DEVICE)
        pr = D['pr'][sl].to(DEVICE)
        y = D['y'][sl].to(DEVICE)
        fid = D['fid'][sl].to(DEVICE)
        vd, mag = model(vp, vc, fid)
        lg = logits_from(vd, pr, model.logit_scale)
        loss = ce_fn(lg, y)
        tot += loss.item() * len(y)
        n += len(y)
        all_y += y.cpu().tolist()
        all_p += lg.argmax(1).cpu().tolist()
    y_true = np.array(all_y)
    y_pred = np.array(all_p)
    macro = f1_score(y_true, y_pred, labels=[0, 1, 2], average='macro', zero_division=0)
    per = f1_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {
        'ce_loss': tot / max(n, 1),
        'macro_f1': float(macro),
        'f1_worsened': float(per[0]),
        'f1_stable': float(per[1]),
        'f1_improved': float(per[2]),
    }

def _train_one(use_ce, use_mag, use_supcon, seed):
    _set_seed(seed)
    model = _make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce_fn = nn.CrossEntropyLoss(weight=W_cls.to(DEVICE))
    best, best_state, best_ep, bad = -1.0, None, 0, 0
    n_train = len(DATA['train']['y'])

    for ep in range(1, ABLATION_EPOCHS + 1):
        model.train()
        tot, n = 0.0, 0
        for _ in range(STEPS):
            if use_supcon:
                idxs = sample_contrastive_batch(TRAIN_BUCKETS)
                if len(idxs) < 4:
                    idxs = random.sample(range(n_train), min(MAX_BATCH_SIZE, n_train))
            else:
                idxs = random.sample(range(n_train), min(MAX_BATCH_SIZE, n_train))

            idxs_t = torch.tensor(idxs, dtype=torch.long)
            vp = DATA['train']['vp'][idxs_t].to(DEVICE)
            vc = DATA['train']['vc'][idxs_t].to(DEVICE)
            pr = DATA['train']['pr'][idxs_t].to(DEVICE)
            y = DATA['train']['y'][idxs_t].to(DEVICE)
            fid = DATA['train']['fid'][idxs_t].to(DEVICE)
            tt = DATA['train']['tt'][idxs_t].to(DEVICE)

            vd, mag = model(vp, vc, fid)
            lg = logits_from(vd, pr, model.logit_scale)

            loss = vd.new_zeros(())
            if use_ce:
                loss = loss + LAMBDA_CE * ce_fn(lg, y)
            if use_mag and mag is not None:
                is_change = (y != C2I['stable']).float()
                loss = loss + LAMBDA_MAG * F.binary_cross_entropy_with_logits(mag, is_change)
            if use_supcon:
                l_con, _st = masked_crossmodal_supcon_loss(
                    vd, tt, y, fid, model.tau_con(), symmetric=_SYM)
                loss = loss + LAMBDA_CON * l_con

            if not torch.isfinite(loss) or (
                loss.item() == 0.0 and not (use_ce or use_mag or use_supcon)
            ):
                loss = vd.sum() * 0.0

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss.item()) * len(idxs)
            n += len(idxs)

        tune = _evaluate(model, 'tune')
        vf1 = tune['macro_f1']
        if vf1 > best:
            best = vf1
            best_ep = ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if ep % 10 == 0 or ep == 1:
            print('  ep %3d  loss %.3f  tuneF1 %.3f  best %.3f' % (
                ep, tot / max(n, 1), vf1, best))

        if bad >= ABLATION_PATIENCE:
            print('  early stop @ ep %d  best tune macro-F1 %.3f' % (ep, best))
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    return model, best_state, best, best_ep, ep


combos = list(product([0, 1], [0, 1], [0, 1]))
if SKIP_ALL_OFF:
    combos = [c for c in combos if c != (0, 0, 0)]
if QUICK_ABLATION:
    combos = [c for c in combos if c[0] == 1]

print('Running %d combos | epochs<=%d patience=%d' % (
    len(combos), ABLATION_EPOCHS, ABLATION_PATIENCE))
print('combos (CE, Mag, SupCon):', combos)
print('contrastive = CROSS-MODAL d<->text | symmetric=', _SYM)

rows = []
for run_i, (ce, mag, sup) in enumerate(combos):
    tag = 'ce%d_mag%d_supcon%d' % (ce, mag, sup)
    print('')
    print('=' * 60)
    print('[%d/%d] %s' % (run_i + 1, len(combos), tag))
    print('=' * 60)

    seed = ABLATION_SEED + run_i
    t0 = time.time()
    model, state, best_tune, best_ep, epochs_ran = _train_one(
        use_ce=bool(ce), use_mag=bool(mag), use_supcon=bool(sup), seed=seed)
    tune_m = _evaluate(model, 'tune')
    test_m = _evaluate(model, 'test') if RUN_TEST_EACH else None
    elapsed = time.time() - t0

    row = {
        'ce': ce, 'mag': mag, 'supcon': sup, 'tag': tag, 'seed': seed,
        'best_epoch': best_ep, 'epochs_ran': epochs_ran,
        'seconds': round(elapsed, 1),
        'tune_macro_f1': tune_m['macro_f1'],
        'tune_f1_worsened': tune_m['f1_worsened'],
        'tune_f1_stable': tune_m['f1_stable'],
        'tune_f1_improved': tune_m['f1_improved'],
    }
    if test_m is not None:
        row.update({
            'test_macro_f1': test_m['macro_f1'],
            'test_f1_worsened': test_m['f1_worsened'],
            'test_f1_stable': test_m['f1_stable'],
            'test_f1_improved': test_m['f1_improved'],
        })
    rows.append(row)

    if SAVE_CKPTS:
        path = '%s/proposed_ablation_%s_seed%d_%s.pt' % (ABL_DIR, tag, seed, ts)
        torch.save({
            'model': state,
            'tune_macro_f1': best_tune,
            'row': row,
            'findings': FINDINGS,
            'classes': CLASSES,
            'config': dict(
                FINDING_CONDITIONING=FINDING_CONDITIONING,
                CONTRASTIVE_SYMMETRIC=_SYM,
                PROTO_SOURCE=PROTO_SOURCE,
                LAMBDA_CE=LAMBDA_CE, LAMBDA_MAG=LAMBDA_MAG, LAMBDA_CON=LAMBDA_CON,
                TAU_CON_INIT=TAU_CON_INIT,
                contrastive='crossmodal_d_to_text',
            ),
        }, path)
        print('  saved', path)

    msg = '  tune macro-F1=%.3f' % tune_m['macro_f1']
    if test_m is not None:
        msg += '  |  test macro-F1=%.3f' % test_m['macro_f1']
    msg += '  |  %.1f min' % (elapsed / 60.0)
    print(msg)

df = pd.DataFrame(rows)
sort_col = 'test_macro_f1' if 'test_macro_f1' in df.columns else 'tune_macro_f1'
df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)

print('')
print('=' * 60)
print('LOSS ABLATION SUMMARY (sorted by %s)' % sort_col)
print('=' * 60)
show_cols = [c for c in [
    'ce', 'mag', 'supcon', 'tune_macro_f1', 'test_macro_f1',
    'test_f1_worsened', 'test_f1_stable', 'test_f1_improved',
    'best_epoch', 'epochs_ran', 'seconds',
] if c in df.columns]
try:
    display(df[show_cols])
except NameError:
    print(df[show_cols].to_string(index=False))

if SAVE_CSV:
    csv_path = '%s/loss_ablation_%s.csv' % (ABL_DIR, ts)
    df.to_csv(csv_path, index=False)
    print('wrote', csv_path)

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = ['CE%d/M%d/S%d' % (r.ce, r.mag, r.supcon) for _, r in df.iterrows()]
    vals = df[sort_col].values
    ax.bar(range(len(vals)), vals, color='steelblue')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel(sort_col)
    ax.set_title('Loss ablation (cross-modal SupCon)')
    ax.set_ylim(0, max(0.5, float(vals.max()) + 0.05))
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('plot skipped:', e)

ABLATION_DF = df
print('Done. Results in ABLATION_DF')

Running 7 combos | epochs<=120 patience=20
combos (CE, Mag, SupCon): [(0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]
contrastive = CROSS-MODAL d<->text | symmetric= False

[1/7] ce0_mag0_supcon1
  ep   1  loss 1.239  tuneF1 0.379  best 0.379
  ep  10  loss 1.152  tuneF1 0.431  best 0.433
  ep  20  loss 1.124  tuneF1 0.434  best 0.474
  ep  30  loss 1.106  tuneF1 0.413  best 0.481
  ep  40  loss 1.084  tuneF1 0.442  best 0.481
  early stop @ ep 41  best tune macro-F1 0.481
  saved /content/drive/MyDrive/3dCT/ctclip_cache/ablations/proposed_ablation_ce0_mag0_supcon1_seed0_20260817_072340.pt
  tune macro-F1=0.481  |  test macro-F1=0.473  |  0.5 min

[2/7] ce0_mag1_supcon0
  ep   1  loss 0.312  tuneF1 0.279  best 0.279
  ep  10  loss 0.191  tuneF1 0.292  best 0.292
  ep  20  loss 0.133  tuneF1 0.272  best 0.305
  ep  30  loss 0.092  tuneF1 0.273  best 0.305
  ep  40  loss 0.057  tuneF1 0.305  best 0.317
  ep  50  loss 0.052  tuneF1 0.268  best 0.332
  ep  60  

   ce  mag  supcon  tune_macro_f1  test_macro_f1  test_f1_worsened  \
0   1    1       0       0.552227       0.575441          0.554855   
1   1    0       0       0.546974       0.561857          0.502778   
2   1    1       1       0.517269       0.473275          0.477791   
3   0    0       1       0.481146       0.473174          0.443857   
4   1    0       1       0.509679       0.454067          0.461153   
5   0    1       1       0.473299       0.436481          0.418109   
6   0    1       0       0.331826       0.346445          0.412617   

   test_f1_stable  test_f1_improved  best_epoch  epochs_ran  seconds  
0        0.483063          0.688406           2          22     12.9  
1        0.491852          0.690940           1          21     11.9  
2        0.375169          0.566866          46          66     47.7  
3        0.362819          0.612847          21          41     27.7  
4        0.357527          0.543520          16          36     25.2  
5        0.38

wrote /content/drive/MyDrive/3dCT/ctclip_cache/ablations/loss_ablation_20260817_072340.csv


<Figure size 800x400 with 1 Axes>

Done. Results in ABLATION_DF
